In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Carga de fuentes de datos
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv("/datasets/users_latam.csv")
usage = pd.read_csv("/datasets/usage.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/plans.csv'

In [ ]:
# Inspección de estructuras iniciales
plans.head()
users.head()
usage.head()

In [ ]:
print("Dimensiones de plans:", plans.shape)
print("Dimensiones de users:", users.shape)
print("Dimensiones de usage:", usage.shape)

plans.info()
users.info()
usage.info()

In [ ]:
# Evaluación de nulos en el perfil de usuarios
print("Valores absolutos:\n", users.isna().sum())
print("\nProporción relativa:\n", users.isna().mean())

In [ ]:
# Evaluación de nulos en el registro de uso
print("Valores absolutos:\n", usage.isna().sum())
print("\nProporción relativa:\n", usage.isna().mean())

Diagnóstico Preliminar de Nulos:
users: La columna city presenta un 11% de valores ausentes que requieren evaluación. La columna churn_date tiene un 88% de nulos, lo cual es lógicamente correcto ya que representa a los clientes que se mantienen activos en la compañía.

usage: La columna date presenta una ausencia marginal (0.12%). Las columnas duration (55%) y length (44%) sugieren una estructura de datos faltantes condicionales al tipo de servicio.

In [ ]:
# Auditoría de consistencia en variables de usuarios
print("Sentinels en user_id:", users["user_id"].isin([-999, 999]).sum())
print("Negativos en user_id:", users["user_id"].le(0).sum())
print("ID Duplicados:", users["user_id"].duplicated().sum())

print("-" * 40)

print("Sentinels en edad (-999):", users["age"].isin([-999, 999]).sum())
print("Edades negativas:", users["age"].le(0).sum())
print("Registros sobre el rango biológico (>=150 años):", users["age"].ge(150).sum())

In [ ]:
# Auditoría de consistencia en el registro transaccional de uso
print("Sentinels en ID de uso:", usage["id"].isin([-999, 999]).sum())
print("IDs de uso duplicados:", usage["id"].duplicated().sum())

print("-" * 40)

print("Sentinels en user_id (usage):", usage["user_id"].isin([-999, 999]).sum())
print("Usuarios únicos en transacciones:", usage["user_id"].nunique())

In [ ]:
# Auditoría de variables categóricas en usuarios
columnas_categoricas = ['city', 'plan']
print(users[columnas_categoricas].value_counts())
print("\nRegistros con caracteres de escape o incógnitas (?):")
print(users[columnas_categoricas].isin(["?", "NA", "UNKNOWN"]).sum())

In [ ]:
# Auditoría de variables categóricas en uso
print(usage['type'].value_counts())
print("Valores inválidos en tipo de uso:", usage['type'].isin(["?", "NA", "UNKNOWN"]).sum())

Diagnóstico de Sentinels e Inconsistencias:
La columna age posee un 1.3% de registros con el valor sentinel -999.

La columna city presenta un 2.4% de registros capturados con el carácter ?.

La tabla usage muestra una alta frecuencia de duplicados en user_id, lo cual es correcto debido a la naturaleza transaccional del consumo de servicios por un mismo cliente.

In [ ]:
# Estandarización de formatos temporales
users['reg_date'] = pd.to_datetime(users["reg_date"])
usage['date'] = pd.to_datetime(usage['date'])

In [ ]:
# Validación cronológica en registros de usuarios
users['reg_date'].dt.year.value_counts().sort_index()

In [ ]:
# Validación cronológica en registros de consumo
usage['date'].dt.year.value_counts().sort_index()

Diagnóstico de Fechas:
Se detectaron 40 registros en reg_date con el año erróneo 2026. Al ser fechas futuras inconsistentes con el límite de 2024, se deben clasificar como errores de captura.

La temporalidad de la tabla usage se encuentra correctamente acotada dentro del año 2024.

In [ ]:
# Imputación de edad mediante la mediana de la población válida
age_mediana = users.loc[users["age"] != -999, "age"].median()
users['age'] = users["age"].replace(-999, age_mediana)

# Estandarización de incógnitas en ciudades a valores nulos reconocidos por Pandas
users["city"] = users["city"].replace("?", pd.NA)

# Tratamiento de anomalías cronológicas futuras (Fechas de registro en 2026)
max_fecha_limite = pd.Timestamp("2024-12-31")
users.loc[users["reg_date"] > max_fecha_limite, "reg_date"] = pd.NaT

In [ ]:
# Análisis de nulos en duración agrupados por tipo de servicio
print("Proporción de nulos en duration por servicio:")
print(usage["duration"].isna().groupby(usage["type"]).mean())

print("\n" + "-"*40 + "\n")

# Análisis de nulos en longitud de texto agrupados por tipo de servicio
print("Proporción de nulos en length por servicio:")
print(usage["length"].isna().groupby(usage["type"]).mean())

Conclusión de QA: Se confirma el mecanismo MAR (Missing at Random). El 100% de las ausencias en duration corresponden a registros de tipo text, mientras que el 100% de las ausencias en length corresponden a registros de tipo call. Los nulos son consistentes y se conservan para no romper la integridad conceptual.

In [ ]:
# Creación de indicadores bandera para conteos eficientes
usage["is_text"] = (usage["type"] == "text").astype(int)
usage["is_call"] = (usage["type"] == "call").astype(int)

# Conversión de segundos a minutos para la métrica de consumo telefónico
usage["call_minutes"] = (usage["duration"] / 60).fillna(0)

# Mapeo descriptivo de columnas de uso
usage["cant_mensajes"] = usage["is_text"]
usage["cant_llamadas"] = usage["is_call"]
usage["cant_minutos_llamada"] = usage["call_minutes"]

# Agregación por entidad única de usuario
usage_agg = usage.groupby('user_id')[['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']].sum().reset_index()

# Integración de métricas de consumo con el perfil maestro de clientes (User Profile)
user_profile = users.merge(usage_agg, on="user_id", how="left")
user_profile.head(5)

In [ ]:
# Análisis descriptivo de variables cuantitativas de consumo y perfil
variables_clave = ["age", "cant_mensajes", "cant_llamadas", "cant_minutos_llamada"]
user_profile[variables_clave].describe()

In [ ]:
# Análisis de participación de mercado por tipo de plan contratado
user_profile["plan"].value_counts(normalize=True) * 100

In [ ]:
# Configuración estética global para el reporte visual
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribución de Edades
sns.histplot(data=user_profile, x="age", hue="plan", kde=True, multiple="stack", ax=axes[0, 0], palette="muted")
axes[0, 0].set_title("Distribución de Edad de los Usuarios por Plan")
axes[0, 0].set_xlabel("Edad")
axes[0, 0].set_ylabel("Frecuencia")

# 2. Distribución de Mensajes Enviados
sns.histplot(data=user_profile, x="cant_mensajes", hue="plan", kde=True, multiple="stack", ax=axes[0, 1], palette="muted")
axes[0, 1].set_title("Distribución de Mensajes Enviados por Plan")
axes[0, 1].set_xlabel("Cantidad de Mensajes")
axes[0, 1].set_ylabel("Frecuencia")

# 3. Distribución de Llamadas Realizadas
sns.histplot(data=user_profile, x="cant_llamadas", hue="plan", kde=True, multiple="stack", ax=axes[1, 0], palette="muted")
axes[1, 0].set_title("Distribución de Llamadas Realizadas por Plan")
axes[1, 0].set_xlabel("Cantidad de Llamadas")
axes[1, 0].set_ylabel("Frecuencia")

# 4. Distribución de Minutos de Llamada Consumidos
sns.histplot(data=user_profile, x="cant_minutos_llamada", hue="plan", kde=True, multiple="stack", ax=axes[1, 1], palette="muted")
axes[1, 1].set_title("Distribución de Minutos de Llamada por Plan")
axes[1, 1].set_xlabel("Minutos Consumidos")
axes[1, 1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()